In [1]:
"""
find_common_genes.py
--------------------
Finds common gene features across SVM-RFE output CSV files for given GSE IDs.

Usage:
  Edit the GSE_IDS and OUTPUTS_DIR variables below, then run:
    python find_common_genes.py
"""

import os
import sys
import pandas as pd

# ============================================================
# CONFIGURATION — edit these
# ============================================================
GSE_IDS     = ["gse37250", "gse37250_hiv"]          # add as many as needed
OUTPUTS_DIR = "outputs"                         # root folder containing per-GSE subfolders
OUTPUT_FILE = f"{OUTPUTS_DIR}/{('_'.join(GSE_IDS))}_common_values.txt"

# ============================================================


def find_svm_rfe_file(gse_id: str) -> str:
    """Locate the svm_rfe_selected_features CSV for a given GSE ID."""
    folder = os.path.join(OUTPUTS_DIR, gse_id)
    if not os.path.isdir(folder):
        print(f"[ERROR] Output folder not found: '{folder}'")
        sys.exit(1)

    candidates = [
        f for f in os.listdir(folder)
        if f.endswith(".csv") and "svm_rfe" in f.lower()
    ]

    if len(candidates) == 0:
        print(f"[ERROR] No SVM-RFE CSV found in '{folder}'")
        print(f"        Files present: {os.listdir(folder)}")
        sys.exit(1)
    if len(candidates) > 1:
        print(f"[WARNING] Multiple SVM-RFE CSVs found in '{folder}', using first: {candidates[0]}")
        print(f"          All candidates: {candidates}")

    return os.path.join(folder, candidates[0])


def load_genes(path: str) -> set:
    """Read a CSV and return the set of gene names (column headers, excluding CLASS)."""
    print(f"  Loading: {path}")
    df = pd.read_csv(path)
    print(f"    Shape: {df.shape}  —  columns (first 5): {df.columns.tolist()[:5]}")

    genes = [c for c in df.columns if c.upper() != "CLASS"]
    print(f"    Genes found: {len(genes)}")
    return set(genes)


def main():
    if len(GSE_IDS) < 2:
        print("[ERROR] Provide at least 2 GSE IDs in the GSE_IDS list.")
        sys.exit(1)

    print("=" * 50)
    print("STEP 1: LOCATING FILES")
    print("=" * 50)
    paths = {}
    for gse_id in GSE_IDS:
        path = find_svm_rfe_file(gse_id)
        paths[gse_id] = path
        print(f"  [{gse_id}] -> {path}")

    print()
    print("=" * 50)
    print("STEP 2: LOADING GENES")
    print("=" * 50)
    gene_sets = {}
    for gse_id, path in paths.items():
        gene_sets[gse_id] = load_genes(path)

    print()
    print("=" * 50)
    print("STEP 3: FINDING COMMON GENES")
    print("=" * 50)
    common = sorted(set.intersection(*gene_sets.values()))
    print(f"  Common genes across {len(GSE_IDS)} datasets: {len(common)}")
    if len(common) == 0:
        print("  [WARNING] No common genes found — check that gene name formats match across files.")
    else:
        print(f"  First 10: {common[:10]}")

    print()
    print("=" * 50)
    print("STEP 4: WRITING OUTPUT")
    print("=" * 50)
    with open(OUTPUT_FILE, "w") as f:
        f.write("\n".join(common))
    print(f"  Written to '{OUTPUT_FILE}'")
    print()
    print("Done.")


if __name__ == "__main__":
    main()

STEP 1: LOCATING FILES
  [gse37250] -> outputs\gse37250\gse37250_svm_rfe_selected_features.csv
  [gse37250_hiv] -> outputs\gse37250_hiv\gse37250_hiv_distinction_svm_rfe_selected_features.csv

STEP 2: LOADING GENES
  Loading: outputs\gse37250\gse37250_svm_rfe_selected_features.csv
    Shape: (537, 13)  —  columns (first 5): ['GBP5', 'DEFA3', 'DEFA1B', 'SEPT4', 'SLC26A8']
    Genes found: 12
  Loading: outputs\gse37250_hiv\gse37250_hiv_distinction_svm_rfe_selected_features.csv
    Shape: (195, 21)  —  columns (first 5): ['HERC5', 'IFI27', 'OAS2', 'ISG15', 'HERC6']
    Genes found: 20

STEP 3: FINDING COMMON GENES
  Common genes across 2 datasets: 2
  First 10: ['IFI27', 'IFI44L']

STEP 4: WRITING OUTPUT
  Written to 'outputs/gse37250_gse37250_hiv_common_values.txt'

Done.
